# **4. Deployment (Web App)**

This notebook creates a simple Flask app for disaster tweet classification.

What this notebook does:
- Loads the saved model
- Defines the same preprocessing logic used in training
- Builds a prediction function
- Writes `app.py`
- Writes HTML templates
- Shows how to run the app locally

In [1]:
from pathlib import Path
APP_DIR = Path("web_app")
TEMPLATE_DIR = APP_DIR / "templates"
STATIC_DIR = APP_DIR / "static"

TEMPLATE_DIR.mkdir(parents=True, exist_ok=True)
STATIC_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
app_py = """
import re
from pathlib import Path

import joblib
import pandas as pd
from flask import Flask, render_template, request

BASE_DIR = Path(__file__).resolve().parent
MODEL_PATH = BASE_DIR.parent / "model_store" / "best_disaster_tweet_model.joblib"

app = Flask(__name__)
model = joblib.load(MODEL_PATH)

def clean_text(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r"http\\S+|www\\S+", " ", text)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"@\\w+", " ", text)
    text = text.replace("#", " ")
    text = re.sub(r"[^a-z0-9\\s]", " ", text)
    text = re.sub(r"\\s+", " ", text).strip()
    return text

def build_features(user_text: str) -> pd.DataFrame:
    cleaned = clean_text(user_text)
    features = pd.DataFrame([{
        "clean_text": cleaned,
        "text_length_chars": len(str(user_text)),
        "word_count": len(cleaned.split()),
        "has_hashtag": int("#" in str(user_text)),
        "has_mention": int("@" in str(user_text)),
        "has_url": int(("http" in str(user_text).lower()) or ("www" in str(user_text).lower()))
    }])
    return features

@app.route("/", methods=["GET", "POST"])
def home():
    prediction = None
    confidence = None
    tweet_text = ""

    if request.method == "POST":
        tweet_text = request.form.get("tweet_text", "").strip()
        if tweet_text:
            features = build_features(tweet_text)
            pred = int(model.predict(features)[0])
            prediction = "Disaster Tweet" if pred == 1 else "Non-Disaster Tweet"

            if hasattr(model, "predict_proba"):
                confidence = float(model.predict_proba(features)[0].max())
            elif hasattr(model, "decision_function"):
                score = float(model.decision_function(features)[0])
                confidence = 1 / (1 + pow(2.71828, -abs(score)))
            else:
                confidence = None

    return render_template(
        "index.html",
        prediction=prediction,
        confidence=confidence,
        tweet_text=tweet_text
    )

if __name__ == "__main__":
    app.run(debug=True)
"""
(APP_DIR / "app.py").write_text(app_py, encoding="utf-8")
print("Created:", APP_DIR / "app.py")

Created: web_app\app.py


In [3]:
index_html = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Disaster Tweet Classifier</title>
    <style>
        body {
            font-family: Arial, sans-serif;
            background: #f4f7fb;
            margin: 0;
            padding: 0;
        }
        .container {
            max-width: 800px;
            margin: 50px auto;
            background: white;
            padding: 30px;
            border-radius: 14px;
            box-shadow: 0 8px 24px rgba(0,0,0,0.08);
        }
        h1 {
            margin-top: 0;
            color: #1f2937;
        }
        p {
            color: #4b5563;
        }
        textarea {
            width: 100%;
            min-height: 160px;
            padding: 14px;
            border: 1px solid #d1d5db;
            border-radius: 10px;
            font-size: 15px;
            resize: vertical;
        }
        button {
            margin-top: 16px;
            background: #2563eb;
            color: white;
            border: none;
            padding: 12px 20px;
            border-radius: 10px;
            cursor: pointer;
            font-size: 15px;
        }
        .result {
            margin-top: 24px;
            padding: 18px;
            border-radius: 12px;
            background: #eef2ff;
        }
        .label {
            font-size: 20px;
            font-weight: bold;
            color: #111827;
        }
    </style>
</head>
<body>
    <div class="container">
        <h1>Disaster Tweet Classification</h1>
        <p>Paste a tweet below and click predict.</p>

        <form method="POST">
            <textarea name="tweet_text" placeholder="Enter tweet text here...">{{ tweet_text }}</textarea>
            <br>
            <button type="submit">Predict</button>
        </form>

        {% if prediction %}
        <div class="result">
            <div class="label">Prediction: {{ prediction }}</div>
            {% if confidence %}
            <p>Confidence: {{ '%.2f'|format(confidence * 100) }}%</p>
            {% endif %}
        </div>
        {% endif %}
    </div>
</body>
</html>
"""
(TEMPLATE_DIR / "index.html").write_text(index_html, encoding="utf-8")
print("Created:", TEMPLATE_DIR / "index.html")

Created: web_app\templates\index.html


In [4]:
requirements_txt = """
flask
pandas
numpy
scikit-learn
joblib
"""
(APP_DIR / "requirements.txt").write_text(requirements_txt.strip() + "\n", encoding="utf-8")
print("Created:", APP_DIR / "requirements.txt")

Created: web_app\requirements.txt


## **How to run the app locally**

**From the project folder, run:**

```bash
cd web_app
pip install -r requirements.txt
python app.py
```

Then open the local URL shown in the terminal.

## **Suggested project structure**

```text
project/
│
├── artifacts/
├── model_store/
├── reports/
├── web_app/
│   ├── app.py
│   ├── requirements.txt
│   └── templates/
│       └── index.html
├── 1_Data_Exploration_Preparation.ipynb
├── 2_Feature_Engineering_Model_Training.ipynb
├── 3_Model_Evaluation.ipynb
└── 4_Deployment_Web_App.ipynb
```

# **Project Overview**

**This project builds a real-time NLP-based web application that predicts whether a given tweet is:**

🔴 Disaster-related (Real Emergency)

🟢 Non-disaster (Normal/Metaphorical Tweet)

The application uses a trained Machine Learning model along with text preprocessing and feature engineering techniques to classify tweets accurately.

## **Objective of Deployment**

**The goal of this deployment is to:**

Provide a user-friendly interface
Allow users to input any tweet
Instantly predict disaster or non-disaster
Show probability score
Display processed details for transparency

## **Streamlit Web App Interface**

The application is built using the Streamlit library, which allows quick and interactive UI development.

## **What it does:**
Displays the main heading of the app
Clearly communicates the purpose of the application
UI Output:

A bold title at the top:

Disaster Tweet Classifier

## **Description Section**
## **Purpose:**
Guides the user on how to use the app
Explains the prediction task

## **Example Tweet Selector**

example_tweets = [
    "Massive fire breaks out in market area, people are running for safety.",
    "I am drowning in assignments this week ",
    "Earthquake felt in Delhi NCR just now.",
    "My phone battery died, what a disaster!"
]

selected_example = st.selectbox(
    "Try an example tweet:",
    options=[""] + example_tweets
)

## **What it does:**
Provides predefined sample tweets

Helps users quickly test the model

## **Benefit:**
Improves usability

Demonstrates real vs fake disaster examples

## **Input Text Area**

tweet_input = st.text_area(
    "Enter tweet text:",
    value=selected_example,
    height=150,
    placeholder="Type or paste a tweet here..."
)

## **Purpose:**

Main input field for user

Accepts custom tweet text

UI:
Large text box

Supports manual typing or pasting

## **Buttons Section**

col1, col2 = st.columns(2)

with col1:
    predict_button = st.button("Predict", use_container_width=True)

with col2:
    clear_button = st.button("Clear", use_container_width=True)

## **Buttons:**
▶ Predict Button

Triggers model prediction

🔄 Clear Button

Resets the input field

if clear_button:
    st.rerun()

## **Backend Processing Flow**

When user clicks Predict, the following steps happen

## **Cleaning includes:**

Convert to lowercase

Remove URLs

Remove mentions (@user)

Remove special characters

Normalize spaces

## **Example:**

Original: Fire in Delhi!!! 😢 http://link

Cleaned: fire in delhi

## **Feature Engineering**

| Feature      | Description       |
| ------------ | ----------------- |
| tweet_length | Length of tweet   |
| word_count   | Number of words   |
| has_hashtag  | Presence of #     |
| has_mention  | Presence of @     |
| has_url      | Presence of links |


## **Data Formatting**

input_df = pd.DataFrame([{
    "clean_text": cleaned,
    **manual_features
}])

## **Purpose:**

Convert input into DataFrame format

Match training structure

## **Applying Preprocessing Pipeline**

transformed_input = preprocessor.transform(input_df)

## **Includes:**
TF-IDF transformation

Feature scaling (if applied)

## **Prediction**
prediction = model.predict(transformed_input)[0]

## **Output:**

1 → Disaster

0 → Non-disaster

## **Probability Score**

probability = model.predict_proba(transformed_input)[0][1]

## **Purpose:**
Shows confidence of prediction

Improves interpretability

## **Output Display (UI Results)**
## **Prediction Result**

if pred == 1:

    st.error("This tweet is likely related to a REAL DISASTER.")

else:

    st.success("This tweet is likely NOT related to a real disaster.")

## **UI Behavior:**
🔴 Red alert → Disaster

🟢 Green message → Non-disaster

## **Probability Display**
st.metric(

          label="Disaster Probability",

          value=f"{prob:.2%}"
    
           )

## **Model Artifacts Used**

The app loads the following files:

best_disaster_model.pkl
preprocessor.pkl
feature_columns.pkl

| File                | Use               |
| ------------------- | ----------------- |
| model.pkl           | trained ML model  |
| preprocessor.pkl    | TF-IDF + pipeline |
| feature_columns.pkl | input structure   |


st.error("App could not load the model artifacts.")

## **Handles:**
Missing model files

Incorrect paths

loading issues

## **How to Run the App**

## **Step 1: Install Dependencies**
pip install -r requirements.txt

cd web_app/

pip install -r requirements.txt

## **Step 2: Run Streamlit App**
streamlit run app.py

## **Step 3: Open in Browser**
Streamlit automatically opens:

http://localhost:8501

## **Key Features of the App**

✔ Real-time tweet classification

✔ Clean and interactive UI

✔ Supports manual & example inputs

✔ Displays prediction confidence

✔ Shows internal processing (transparency)

✔ Built with scalable architecture